# 🧠 NanoReason-3B — GSM8K Evaluation (Transformers)
Đánh giá **GSM8K** cho **4 models**:

| # | Model | Loại |
|---|-------|------|
| 1 | `NanoReason-3B` | Full model — **Ours** |
| 2 | `Qwen2.5-3B-SFT-LoRA` | LoRA cần merge — **SFT Baseline** |
| 3 | `Qwen2.5-3B-Instruct` | Pretrained chưa fine-tune — **Pretrained Baseline** |
| 4 | `DeepSeek-R1-Distill-1.5B` | Model nhỏ hơn — **External Baseline** |

> ✅ **Dùng HuggingFace Transformers thay vLLM** — ổn định hơn trên T4

## Cell 1 — Cài đặt thư viện

In [1]:
import subprocess, sys

packages = [
    "transformers",
    "peft",
    "accelerate",
    "datasets",
    "huggingface_hub",
    "pandas",
    "torch",
    "sentencepiece",
    "protobuf",
]

for pkg in packages:
    print(f"📦 Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  ⚠️  {result.stderr[-200:]}")

print("\n✅ Tất cả thư viện đã cài xong!")

📦 Installing transformers...
📦 Installing peft...
📦 Installing accelerate...
📦 Installing datasets...
📦 Installing huggingface_hub...
📦 Installing pandas...
📦 Installing torch...
📦 Installing sentencepiece...
📦 Installing protobuf...

✅ Tất cả thư viện đã cài xong!


## Cell 2 — Config
> ⚠️ **Chỉ sửa phần này** trước khi chạy

In [2]:
import os, gc, re, time, json, traceback
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM

# ──────────────────────────────────────────────────────────────────────
# ⚠️ CẬP NHẬT CÁC PATH BÊN DƯỚI
# ──────────────────────────────────────────────────────────────────────

# Model 1: NanoReason-3B — full model hoàn chỉnh (không cần merge)
NANOREASON_PATH = "/kaggle/input/datasets/ductaiphan/nanoreason-3b/NanoReason-3B-Final"

# Model 2: SFT LoRA — chỉ chứa adapter weights
SFT_LORA_PATH   = "/kaggle/input/datasets/thinguynaas/nckh-week6-baseline-standard-lora/Standard_LoRA/checkpoint-2255"

# Path lưu SFT model sau khi merge (sẽ tự tạo ở Cell 8)
SFT_MERGED_PATH = "/kaggle/working/sft_qwen_merged"

# ──────────────────────────────────────────────────────────────────────

# None = full test set (1319 câu), đặt số nhỏ hơn (vd: 50) để test nhanh
GSM8K_N = None

# Batch size — T4 15.6GB: 4 là an toàn cho 3B model
BATCH_SIZE = 4

# ──────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Config loaded")
print(f"   NanoReason path  : {NANOREASON_PATH}")
print(f"   SFT LoRA path    : {SFT_LORA_PATH}")
print(f"   SFT merged path  : {SFT_MERGED_PATH}")
print(f"   GSM8K samples    : {'Full (1319)' if GSM8K_N is None else GSM8K_N}")
print(f"   Batch size       : {BATCH_SIZE}")
print(f"   Device           : {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU              : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ Config loaded
   NanoReason path  : /kaggle/input/datasets/ductaiphan/nanoreason-3b/NanoReason-3B-Final
   SFT LoRA path    : /kaggle/input/datasets/thinguynaas/nckh-week6-baseline-standard-lora/Standard_LoRA/checkpoint-2255
   SFT merged path  : /kaggle/working/sft_qwen_merged
   GSM8K samples    : Full (1319)
   Batch size       : 4
   Device           : cuda
   GPU              : Tesla T4
   VRAM             : 15.6 GB


## Cell 3 — Load GSM8K dataset

In [3]:
print("📂 Loading GSM8K test set...")
gsm8k_data = load_dataset("gsm8k", "main", split="test")

if GSM8K_N:
    gsm8k_data = gsm8k_data.select(range(GSM8K_N))

gsm8k_list = list(gsm8k_data)
print(f"✅ GSM8K test: {len(gsm8k_list)} samples")
print(f"\nSample đầu tiên:")
print(f"  Q: {gsm8k_list[0]['question'][:100]}...")
print(f"  A: {gsm8k_list[0]['answer'].split('####')[-1].strip()}")

📂 Loading GSM8K test set...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

✅ GSM8K test: 1319 samples

Sample đầu tiên:
  Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for ...
  A: 18


## Cell 4 — Helper functions

In [4]:
def extract_answer(text: str) -> str:
    # Priority 1: \\boxed{} — LaTeX format
    boxed = re.findall(r'\\boxed\{([^}]*)\}', text)
    if boxed:
        return boxed[-1].replace(',', '').strip()
    # Priority 2: #### answer — GSM8K native format
    hashes = re.findall(r'####\s*(-?\d[\d,\.]*)', text)
    if hashes:
        return hashes[-1].replace(',', '').strip()
    # Fallback: số cuối trong text
    nums = re.findall(r'-?\d[\d,\.]*', text)
    return nums[-1].replace(',', '').strip() if nums else ""


def is_correct(pred: str, gt: str) -> bool:
    try:
        return abs(float(pred.replace(',', '')) - float(str(gt).replace(',', ''))) < 1e-3
    except (ValueError, TypeError):
        return pred.strip().lower() == str(gt).strip().lower()


def get_gsm8k_gt(sample: dict) -> str:
    return sample['answer'].split('####')[-1].strip().replace(',', '')


# Quick test
assert extract_answer("The answer is \\boxed{42}") == "42"
assert extract_answer("#### 100") == "100"
assert is_correct("42", "42") is True
assert is_correct("42.0", "42") is True
print("✅ Helper functions OK")

✅ Helper functions OK


## Cell 5 — Prompt builders

| Model | Prompt strategy |
|-------|----------------|
| **NanoReason** | `#### UNDERSTAND ####` prefix |
| **SFT LoRA** | Chat template + `\\boxed{}` |
| **Qwen2.5-Instruct** | Chat template + CoT prefix |
| **DeepSeek-R1** | `<think>` tag, không system prompt |

In [5]:
def build_prompt_nanoreason(sample: dict) -> str:
    q = sample['question']
    return (
        "<|im_start|>system\n"
        "You are NanoReason, an expert mathematical reasoning AI. "
        "Think carefully and show your work step by step.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{q}<|im_end|>\n"
        "<|im_start|>assistant\n"
        "#### UNDERSTAND ####\n"
    )


def build_prompt_sft_lora(sample: dict) -> str:
    q = sample['question']
    return (
        "<|im_start|>system\n"
        "You are a helpful math assistant. "
        "Solve the problem step by step, then box your final answer as \\boxed{answer}.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{q}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def build_prompt_qwen_base(sample: dict) -> str:
    q = sample['question']
    return (
        "<|im_start|>system\n"
        "You are a helpful math assistant. "
        "Solve the problem step by step, then write your final answer as \\boxed{answer}.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{q}<|im_end|>\n"
        "<|im_start|>assistant\n"
        "Let me solve this step by step.\n\n"
    )


def build_prompt_deepseek_r1(sample: dict) -> str:
    q = sample['question']
    return (
        "<|im_start|>user\n"
        f"{q}<|im_end|>\n"
        "<|im_start|>assistant\n"
        "<think>\n"
    )


PROMPT_BUILDERS = {
    "nanoreason":  build_prompt_nanoreason,
    "sft_lora":    build_prompt_sft_lora,
    "qwen_base":   build_prompt_qwen_base,
    "deepseek_r1": build_prompt_deepseek_r1,
}

print("📝 Preview prompts (sample 0):\n")
for name, fn in PROMPT_BUILDERS.items():
    print(f"── {name} " + "─"*35)
    print(fn(gsm8k_list[0])[:200])
    print()

📝 Preview prompts (sample 0):

── nanoreason ───────────────────────────────────
<|im_start|>system
You are NanoReason, an expert mathematical reasoning AI. Think carefully and show your work step by step.<|im_end|>
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats thre

── sft_lora ───────────────────────────────────
<|im_start|>system
You are a helpful math assistant. Solve the problem step by step, then box your final answer as \boxed{answer}.<|im_end|>
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eat

── qwen_base ───────────────────────────────────
<|im_start|>system
You are a helpful math assistant. Solve the problem step by step, then write your final answer as \boxed{answer}.<|im_end|>
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She e

── deepseek_r1 ───────────────────────────────────
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the r

## Cell 6 — Cleanup utility

In [6]:
def cleanup_model(model, tokenizer=None):
    """Giải phóng GPU memory sau mỗi model."""
    print("🧹 Releasing GPU memory...")
    del model
    if tokenizer:
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  ✅ GPU memory released — used: {used:.2f}/{total:.1f} GB\n")
    else:
        print("  ✅ Done\n")

print("✅ cleanup_model() ready")

✅ cleanup_model() ready


## Cell 7 — Eval function (Transformers)

In [7]:
def run_gsm8k_eval(
    model_path: str,
    model_name: str,
    prompt_style: str,
    samples: list,
    batch_size: int = BATCH_SIZE,
    max_new_tokens: int = 512,
) -> dict:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"\n{'='*70}")
    print(f"🚀  Model   : {model_name}")
    print(f"    Path    : {model_path}")
    print(f"    Prompt  : {prompt_style}")
    print(f"    Samples : {len(samples)}")
    print(f"    Batch   : {batch_size}")
    print(f"{'='*70}")

    # ── Load model & tokenizer ───────────────────────────────────────
    try:
        print("📥 Loading tokenizer...")
        # ✅ FIX: local_files_only=True cho local path, False cho HuggingFace repo
        _is_local = model_path.startswith("/")
        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            trust_remote_code=True,
            padding_side="left",
            local_files_only=_is_local,
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        print("📥 Loading model (float16)...")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            local_files_only=_is_local,
        )
        model.eval()
        print(f"  ✅ Model loaded — device: {next(model.parameters()).device}")

    except Exception as e:
        print(f"❌ Load model thất bại: {e}")
        traceback.print_exc()
        return {"model": model_name, "gsm8k": "LOAD_ERROR", "gsm8k_raw": str(e), "errors": []}

    # ── Generate theo batch ──────────────────────────────────────────
    builder  = PROMPT_BUILDERS[prompt_style]
    prompts  = [builder(s) for s in samples]
    all_outputs = []

    total_batches = (len(prompts) + batch_size - 1) // batch_size
    print(f"\n🔄 Generating {len(prompts)} samples in {total_batches} batches...")

    try:
        for batch_idx in range(0, len(prompts), batch_size):
            batch_prompts = prompts[batch_idx: batch_idx + batch_size]

            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024,
            ).to(DEVICE)

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,          # greedy — reproducible
                    temperature=1.0,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            # Chỉ lấy phần generated (bỏ phần prompt)
            input_len = inputs["input_ids"].shape[1]
            for out_ids in output_ids:
                gen_text = tokenizer.decode(
                    out_ids[input_len:],
                    skip_special_tokens=True
                )
                all_outputs.append(gen_text)

            # Progress
            done = min(batch_idx + batch_size, len(prompts))
            if (batch_idx // batch_size + 1) % 50 == 0 or done == len(prompts):
                print(f"  [{done}/{len(prompts)}] batches done...")

    except Exception as e:
        print(f"❌ Generation thất bại: {e}")
        traceback.print_exc()
        cleanup_model(model, tokenizer)
        return {"model": model_name, "gsm8k": "GEN_ERROR", "gsm8k_raw": str(e), "errors": []}

    # ── Tính accuracy ────────────────────────────────────────────────
    correct, errors = 0, []
    for i, (sample, gen_text) in enumerate(zip(samples, all_outputs)):
        pred = extract_answer(gen_text)
        gt   = get_gsm8k_gt(sample)
        if is_correct(pred, gt):
            correct += 1
        elif len(errors) < 5:
            errors.append({
                "idx": i,
                "q":   sample['question'][:80],
                "gt":  gt,
                "pred": pred,
                "gen": gen_text[:200],
            })

    accuracy = correct / len(samples) * 100
    cleanup_model(model, tokenizer)

    print(f"🏆  {model_name}: {correct}/{len(samples)} = {accuracy:.2f}%")
    return {
        "model":     model_name,
        "gsm8k":     f"{accuracy:.1f}%",
        "gsm8k_raw": f"{correct}/{len(samples)}",
        "errors":    errors,
    }

print("✅ run_gsm8k_eval() ready")

✅ run_gsm8k_eval() ready


## Cell 8 — Merge SFT LoRA adapter → full model
> vLLM không load được PEFT adapter trực tiếp — cần merge thành full model trước.
> Transformers **có thể load PEFT trực tiếp** nhưng vẫn merge để nhất quán.

**Chỉ cần chạy 1 lần duy nhất.**

In [8]:
from peft import PeftModel

if os.path.exists(SFT_MERGED_PATH):
    print(f"✅ Merged model đã tồn tại → {SFT_MERGED_PATH}")
    print("   Bỏ qua bước merge, tiếp tục Cell 9.")
else:
    print("🔀 Bắt đầu merge SFT LoRA vào base model...")
    print("   (chạy trên CPU — tránh VRAM OOM, ~5-10 phút)\n")

    # 1. Load base model lên CPU
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-3B-Instruct",
        dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )
    print("  ✅ Base model loaded (CPU)")

    # 2. Gắn LoRA adapter
    peft_model = PeftModel.from_pretrained(base_model, SFT_LORA_PATH)
    print("  ✅ LoRA adapter loaded")

    # 3. Merge
    merged_model = peft_model.merge_and_unload()
    print("  ✅ Merge hoàn tất")

    # 4. Save
    os.makedirs(SFT_MERGED_PATH, exist_ok=True)
    merged_model.save_pretrained(SFT_MERGED_PATH)

    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen2.5-3B-Instruct", trust_remote_code=True
    )
    tokenizer.save_pretrained(SFT_MERGED_PATH)

    # 5. Cleanup
    del base_model, peft_model, merged_model, tokenizer
    gc.collect()

    print(f"\n✅ Merged model saved → {SFT_MERGED_PATH}")
    print("   Sẵn sàng để eval ở Cell 10.")

🔀 Bắt đầu merge SFT LoRA vào base model...
   (chạy trên CPU — tránh VRAM OOM, ~5-10 phút)



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


✅ Merged model saved → /kaggle/working/sft_qwen_merged
   Sẵn sàng để eval ở Cell 10.


## Cell 9 — Eval: NanoReason-3B *(Ours — GRPO trained)*

In [9]:
all_results = []

result_nanoreason = run_gsm8k_eval(
    model_path   = NANOREASON_PATH,
    model_name   = "NanoReason-3B (Ours)",
    prompt_style = "nanoreason",
    samples      = gsm8k_list,
)
all_results.append(result_nanoreason)
print(f"\n📊 NanoReason-3B → {result_nanoreason['gsm8k']} ({result_nanoreason['gsm8k_raw']})")


🚀  Model   : NanoReason-3B (Ours)
    Path    : /kaggle/input/datasets/ductaiphan/nanoreason-3b/NanoReason-3B-Final
    Prompt  : nanoreason
    Samples : 1319
    Batch   : 4
📥 Loading tokenizer...
📥 Loading model (float16)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✅ Model loaded — device: cuda:0

🔄 Generating 1319 samples in 330 batches...
  [200/1319] batches done...
  [400/1319] batches done...
  [600/1319] batches done...
  [800/1319] batches done...
  [1000/1319] batches done...
  [1200/1319] batches done...
  [1319/1319] batches done...
🧹 Releasing GPU memory...
  ✅ GPU memory released — used: 3.25/15.6 GB

🏆  NanoReason-3B (Ours): 1038/1319 = 78.70%

📊 NanoReason-3B → 78.7% (1038/1319)


## Cell 10 — Eval: Qwen2.5-3B SFT LoRA *(Standard LoRA Baseline)*
Cùng base model với NanoReason nhưng train bằng **SFT thông thường** (không GRPO).

In [10]:
assert os.path.exists(SFT_MERGED_PATH), (
    f"❌ Merged model chưa tồn tại tại {SFT_MERGED_PATH}\n"
    "   → Chạy Cell 8 trước!"
)

result_sft = run_gsm8k_eval(
    model_path   = SFT_MERGED_PATH,
    model_name   = "Qwen2.5-3B SFT LoRA (Baseline)",
    prompt_style = "sft_lora",
    samples      = gsm8k_list,
)
all_results.append(result_sft)
print(f"\n📊 SFT LoRA → {result_sft['gsm8k']} ({result_sft['gsm8k_raw']})")


🚀  Model   : Qwen2.5-3B SFT LoRA (Baseline)
    Path    : /kaggle/working/sft_qwen_merged
    Prompt  : sft_lora
    Samples : 1319
    Batch   : 4
📥 Loading tokenizer...
📥 Loading model (float16)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  ✅ Model loaded — device: cuda:0

🔄 Generating 1319 samples in 330 batches...
  [200/1319] batches done...
  [400/1319] batches done...
  [600/1319] batches done...
  [800/1319] batches done...
  [1000/1319] batches done...
  [1200/1319] batches done...
  [1319/1319] batches done...
🧹 Releasing GPU memory...
  ✅ GPU memory released — used: 6.18/15.6 GB

🏆  Qwen2.5-3B SFT LoRA (Baseline): 1027/1319 = 77.86%

📊 SFT LoRA → 77.9% (1027/1319)


## Cell 11 — Eval: Qwen2.5-3B-Instruct *(Pretrained Baseline)*

In [11]:
print("📥 Downloading Qwen2.5-3B-Instruct...")
qwen_path = snapshot_download("Qwen/Qwen2.5-3B-Instruct")

result_qwen = run_gsm8k_eval(
    model_path   = qwen_path,
    model_name   = "Qwen2.5-3B-Instruct (Pretrained)",
    prompt_style = "qwen_base",
    samples      = gsm8k_list,
)
all_results.append(result_qwen)
print(f"\n📊 Qwen2.5-3B → {result_qwen['gsm8k']} ({result_qwen['gsm8k_raw']})")

📥 Downloading Qwen2.5-3B-Instruct...


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]


🚀  Model   : Qwen2.5-3B-Instruct (Pretrained)
    Path    : /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1
    Prompt  : qwen_base
    Samples : 1319
    Batch   : 4
📥 Loading tokenizer...
📥 Loading model (float16)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  ✅ Model loaded — device: cuda:0

🔄 Generating 1319 samples in 330 batches...
  [200/1319] batches done...
  [400/1319] batches done...
  [600/1319] batches done...
  [800/1319] batches done...
  [1000/1319] batches done...
  [1200/1319] batches done...
  [1319/1319] batches done...
🧹 Releasing GPU memory...
  ✅ GPU memory released — used: 6.18/15.6 GB

🏆  Qwen2.5-3B-Instruct (Pretrained): 1106/1319 = 83.85%

📊 Qwen2.5-3B → 83.9% (1106/1319)


## Cell 12 — Eval: DeepSeek-R1-Distill-Qwen-1.5B *(External Baseline)*

In [12]:
print("📥 Downloading DeepSeek-R1-Distill-Qwen-1.5B...")
ds_path = snapshot_download("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")

result_deepseek = run_gsm8k_eval(
    model_path   = ds_path,
    model_name   = "DeepSeek-R1-Distill-1.5B",
    prompt_style = "deepseek_r1",
    samples      = gsm8k_list,
)
all_results.append(result_deepseek)
print(f"\n📊 DeepSeek-R1-1.5B → {result_deepseek['gsm8k']} ({result_deepseek['gsm8k_raw']})")

📥 Downloading DeepSeek-R1-Distill-Qwen-1.5B...


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]


🚀  Model   : DeepSeek-R1-Distill-1.5B
    Path    : /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B/snapshots/ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562
    Prompt  : deepseek_r1
    Samples : 1319
    Batch   : 4
📥 Loading tokenizer...
📥 Loading model (float16)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✅ Model loaded — device: cuda:0

🔄 Generating 1319 samples in 330 batches...
  [200/1319] batches done...
  [400/1319] batches done...
  [600/1319] batches done...
  [800/1319] batches done...
  [1000/1319] batches done...
  [1200/1319] batches done...
  [1319/1319] batches done...
🧹 Releasing GPU memory...
  ✅ GPU memory released — used: 3.56/15.6 GB

🏆  DeepSeek-R1-Distill-1.5B: 539/1319 = 40.86%

📊 DeepSeek-R1-1.5B → 40.9% (539/1319)


## Cell 13 — Kết quả cuối & lưu CSV

In [13]:
print("\n" + "=" * 65)
print("📋 FINAL RESULTS — GSM8K")
print("=" * 65)

clean_results = [{k: v for k, v in r.items() if k != "errors"} for r in all_results]
df = pd.DataFrame(clean_results)
print(df.to_string(index=False))

out_csv = "/kaggle/working/eval_gsm8k_results.csv"
df.to_csv(out_csv, index=False)
print(f"\n💾 Saved → {out_csv}")

# ── Paper table ──────────────────────────────────────────────────────
print("\n📄 PAPER TABLE FORMAT:")
print(f"{'Model':<40} {'GSM8K':>10} {'Params':>8} {'Type':>12}")
print("─" * 74)
for r in all_results:
    params = "1.5B" if "1.5B" in r['model'] else "3B"
    mtype  = ("GRPO"      if "Ours"     in r['model'] else
              "SFT LoRA"  if "SFT"      in r['model'] else
              "Distill"   if "DeepSeek" in r['model'] else
              "Pretrain")
    print(f"{r['model']:<40} {r.get('gsm8k','N/A'):>10} {params:>8} {mtype:>12}")

# ── Debug errors ─────────────────────────────────────────────────────
print("\n\n🔍 DEBUG — Mẫu dự đoán sai (tối đa 2 mỗi model):")
for r in all_results:
    if r.get("errors"):
        print(f"\n  [{r['model']}]")
        for e in r["errors"][:2]:
            print(f"    Q    : {e['q']}...")
            print(f"    GT   : {e['gt']}  |  PRED: {e['pred']}")
            print(f"    GEN  : {e['gen'][:150]}")


📋 FINAL RESULTS — GSM8K
                           model gsm8k gsm8k_raw
            NanoReason-3B (Ours) 78.7% 1038/1319
  Qwen2.5-3B SFT LoRA (Baseline) 77.9% 1027/1319
Qwen2.5-3B-Instruct (Pretrained) 83.9% 1106/1319
        DeepSeek-R1-Distill-1.5B 40.9%  539/1319

💾 Saved → /kaggle/working/eval_gsm8k_results.csv

📄 PAPER TABLE FORMAT:
Model                                         GSM8K   Params         Type
──────────────────────────────────────────────────────────────────────────
NanoReason-3B (Ours)                          78.7%       3B         GRPO
Qwen2.5-3B SFT LoRA (Baseline)                77.9%       3B     SFT LoRA
Qwen2.5-3B-Instruct (Pretrained)              83.9%       3B     Pretrain
DeepSeek-R1-Distill-1.5B                      40.9%     1.5B      Distill


🔍 DEBUG — Mẫu dự đoán sai (tối đa 2 mỗi model):

  [NanoReason-3B (Ours)]
    Q    : Josh decides to try flipping a house.  He buys a house for $80,000 and then puts...
    GT   : 70000  |  PRED: 50000
    GEN 

## Cell 14 — Sanity check *(optional)*
So sánh generation của **NanoReason** vs **SFT LoRA** trên cùng 3 mẫu.

In [14]:
# Uncomment để chạy sanity check (load 2 model, ~8 phút)

# from transformers import AutoTokenizer, AutoModelForCausalLM
#
# MODELS_TO_CHECK = [
#     (NANOREASON_PATH, "nanoreason", "NanoReason-3B"),
#     (SFT_MERGED_PATH, "sft_lora",   "SFT LoRA"),
# ]
#
# for model_path, style, name in MODELS_TO_CHECK:
#     print(f"\n{'='*65}")
#     print(f"🔍 {name}")
#     print(f"{'='*65}")
#     tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
#     if tok.pad_token is None:
#         tok.pad_token = tok.eos_token
#     mdl = AutoModelForCausalLM.from_pretrained(
#         model_path, dtype=torch.float16, device_map="auto", trust_remote_code=True
#     ).eval()
#
#     for i in range(3):
#         sample = gsm8k_list[i]
#         prompt = PROMPT_BUILDERS[style](sample)
#         inp    = tok(prompt, return_tensors="pt").to(DEVICE)
#         with torch.no_grad():
#             out = mdl.generate(**inp, max_new_tokens=400, do_sample=False,
#                                pad_token_id=tok.pad_token_id)
#         gen = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
#         gt  = get_gsm8k_gt(sample)
#         pred = extract_answer(gen)
#         status = "✅" if is_correct(pred, gt) else "❌"
#         print(f"\n  Sample {i+1} {status}  GT={gt}  PRED={pred}")
#         print(f"  Q: {sample['question'][:80]}...")
#         print(f"  GEN:\n{gen[:400]}")
#     cleanup_model(mdl, tok)

print("ℹ️  Uncomment code trên để chạy sanity check")

ℹ️  Uncomment code trên để chạy sanity check
